# VoxBind Generated Sample Visualizer

Visualizes the output of `sample.py` — a pocket + its generated ligands — alongside the ground-truth ligand.

| Section | Contents |
|---|---|
| 1. Config | Set sample directory and target index |
| 2. Load | Parse pocket PDB, GT ligand, generated samples |
| 3. 3D view | Interactive Plotly — pocket + GT + all generated ligands |
| 4. 2D grid | RDKit depictions + per-mol QED / SA |
| 5. Summary | Metrics across all generated samples for this target |

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import re
import math
from glob import glob
from pathlib import Path
from io import BytesIO

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, Descriptors, QED, Crippen, Lipinski
from rdkit.Chem.Draw import rdMolDraw2D
from PIL import Image

plt.rcParams.update({"figure.dpi": 120, "font.size": 9})

## 1. Config

Set `SAMPLE_DIR` to the output of `sample.py` and `TARGET_IDX` to the pocket you want to explore.

In [2]:
# ── adjust these two lines ────────────────────────────────────────────────────
SAMPLE_DIR = Path("..") / "voxbind" / "exps" / "exp_sig0.9" / "samples" / "res"
TARGET_IDX = 0           # integer, or None to show a summary of all targets
# ─────────────────────────────────────────────────────────────────────────────

SAMPLE_DIR = Path(SAMPLE_DIR).resolve()
target_dirs = sorted(SAMPLE_DIR.glob("target_*"))
print(f"Sample dir   : {SAMPLE_DIR}")
print(f"Targets found: {len(target_dirs)}")
for td in target_dirs:
    sdf = td / "samples.sdf"
    n = len(list(Chem.SDMolSupplier(str(sdf), sanitize=False))) if sdf.exists() else 0
    print(f"  {td.name}  —  {n} generated mols  —  {td.name}")

Sample dir   : /home/shpark/prj-ligand/Voxbind/voxbind/exps/exp_sig0.9/samples/res
Targets found: 11
  target_00  —  4 generated mols  —  target_00
  target_01  —  4 generated mols  —  target_01
  target_02  —  4 generated mols  —  target_02
  target_03  —  4 generated mols  —  target_03
  target_04  —  4 generated mols  —  target_04
  target_05  —  4 generated mols  —  target_05
  target_06  —  4 generated mols  —  target_06
  target_07  —  4 generated mols  —  target_07
  target_08  —  4 generated mols  —  target_08
  target_09  —  4 generated mols  —  target_09
  target_10  —  4 generated mols  —  target_10


## 2. Load pocket, GT ligand, generated samples

In [3]:
# ── element colour maps ───────────────────────────────────────────────────────
ELEM_COLOR = {
    "C": "gray", "N": "cornflowerblue", "O": "tomato",
    "S": "gold",  "F": "limegreen",     "Cl": "lime",
    "P": "orange","H": "white",
}
ELEM_SIZE  = {"C": 3.5, "N": 3.5, "O": 3.2, "S": 4.5,
              "F": 2.8, "Cl": 4.0, "P": 4.0, "H": 1.8}


def parse_pocket_pdb(pdb_path: str):
    """Return arrays of (coords [N,3], element [N]) from ATOM/HETATM lines."""
    coords, elems = [], []
    for line in open(pdb_path):
        if not (line.startswith("ATOM") or line.startswith("HETATM")):
            continue
        try:
            x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
        except ValueError:
            continue
        # element column (cols 76-78) or derive from atom name
        elem = line[76:78].strip() if len(line) > 76 else ""
        if not elem:
            elem = re.sub(r"[^A-Za-z]", "", line[12:16]).strip()[:2].capitalize()
        # skip hydrogens
        if elem in ("H", "D"):
            continue
        coords.append([x, y, z])
        elems.append(elem)
    return np.array(coords, dtype=float), elems


def mol_coords(mol: Chem.Mol) -> np.ndarray:
    """Return [N,3] heavy-atom coordinates from an RDKit mol."""
    conf = mol.GetConformer()
    return np.array([conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())])


def mol_elems(mol: Chem.Mol) -> list[str]:
    return [a.GetSymbol() for a in mol.GetAtoms()]


def load_target(target_dir: Path):
    """Load pocket PDB, GT ligand SDF, and generated samples.sdf from a target dir."""
    # pocket PDB
    pdbs = sorted(target_dir.glob("*_pocket10.pdb"))
    if not pdbs:
        raise FileNotFoundError(f"No pocket PDB found in {target_dir}")
    pocket_coords, pocket_elems = parse_pocket_pdb(str(pdbs[0]))
    pocket_name = pdbs[0].stem

    # GT ligand (the non-samples SDF file)
    gt_sdfs = [p for p in target_dir.glob("*.sdf") if p.name != "samples.sdf"]
    gt_mol = None
    if gt_sdfs:
        gt_mol = next(iter(Chem.SDMolSupplier(str(gt_sdfs[0]), removeHs=True, sanitize=True)), None)

    # generated samples
    gen_mols = []
    samples_sdf = target_dir / "samples.sdf"
    if samples_sdf.exists():
        for mol in Chem.SDMolSupplier(str(samples_sdf), removeHs=False, sanitize=True):
            if mol is not None:
                gen_mols.append(mol)

    return pocket_coords, pocket_elems, gt_mol, gen_mols, pocket_name


# ── load selected target ──────────────────────────────────────────────────────
if TARGET_IDX is None:
    print("TARGET_IDX is None — run section 5 for multi-target summary.")
else:
    td = target_dirs[TARGET_IDX]
    poc_coords, poc_elems, gt_mol, gen_mols, pocket_name = load_target(td)

    print(f"Target       : {td.name}")
    print(f"Pocket       : {pocket_name}")
    print(f"Pocket atoms : {len(poc_coords)} (heavy only)")
    print(f"GT ligand    : {'found' if gt_mol else 'not found'}",
          f"({gt_mol.GetNumAtoms()} atoms)" if gt_mol else "")
    print(f"Generated    : {len(gen_mols)} molecules")

Target       : target_00
Pocket       : BSD_ASPTE_1_130_0__2z3h_A_rec_1wn6_bst_lig_tt_docked_3_pocket10
Pocket atoms : 409 (heavy only)
GT ligand    : found (31 atoms)
Generated    : 4 molecules


## 3. Interactive 3D Overlay

**Pocket** — grey atoms, semi-transparent.  
**GT ligand** — larger atoms, coloured by element, outlined in white (toggle in legend).  
**Generated ligand** — atoms coloured by element (C/N/O/S/F/Cl/P), with labels; one molecule shown.

In [5]:
palette = px.colors.qualitative.Plotly + px.colors.qualitative.Safe
traces = []

# ── Pocket ────────────────────────────────────────────────────────────────────
for elem in sorted(set(poc_elems)):
    mask = [i for i, e in enumerate(poc_elems) if e == elem]
    xyz  = poc_coords[mask]
    traces.append(go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode="markers",
        marker=dict(
            size=ELEM_SIZE.get(elem, 3),
            color=ELEM_COLOR.get(elem, "white"),
            opacity=0.18,
        ),
        name=f"Pocket {elem}",
        legendgroup="pocket",
        showlegend=True,
    ))

# ── GT ligand ─────────────────────────────────────────────────────────────────
if gt_mol is not None:
    gt_xyz   = mol_coords(gt_mol)
    gt_elems = mol_elems(gt_mol)
    for elem in sorted(set(gt_elems)):
        mask = [i for i, e in enumerate(gt_elems) if e == elem]
        xyz  = gt_xyz[mask]
        traces.append(go.Scatter3d(
            x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
            mode="markers",
            marker=dict(
                size=ELEM_SIZE.get(elem, 3.5) * 1.8,
                color=ELEM_COLOR.get(elem, "purple"),
                opacity=0.95,
                line=dict(width=1.5, color="white"),
                symbol="diamond",
            ),
            name=f"GT {elem}",
            legendgroup="gt",
        ))

# ── Generated samples ─────────────────────────────────────────────────────────
# for i, mol in enumerate(gen_mols):
#     xyz   = mol_coords(mol)
#     elems = mol_elems(mol)
#     smi   = Chem.MolToSmiles(mol)
#     color = palette[i % len(palette)]
#     traces.append(go.Scatter3d(
#         x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
#         mode="markers",
#         marker=dict(
#             size=6,
#             color=color,
#             opacity=0.90,
#             line=dict(width=0.8, color="white"),
#         ),
#         name=f"Gen {i}: {smi[:40]}{'…' if len(smi)>40 else ''}",
#         legendgroup=f"gen_{i}",
#         text=[f"{e}" for e in elems],
#         hovertemplate="%{text}<br>(%{x:.2f}, %{y:.2f}, %{z:.2f})<extra>Gen %{fullData.name}</extra>",
#     ))

# Single sample generation
# i, mol = 0, gen_mols[0]
# xyz   = mol_coords(mol)
# elems = mol_elems(mol)
# smi   = Chem.MolToSmiles(mol)
# # Generated ligand: one trace per element, colored by ELEM_COLOR (same as pocket/GT)
# for elem in sorted(set(elems)):
#     mask = [j for j, e in enumerate(elems) if e == elem]
#     xyz_e = xyz[mask]
#     traces.append(go.Scatter3d(
#         x=xyz_e[:, 0], y=xyz_e[:, 1], z=xyz_e[:, 2],
#         mode="markers+text",
#         marker=dict(
#             size=ELEM_SIZE.get(elem, 3.5) * 1.4,
#             color=ELEM_COLOR.get(elem, "purple"),
#             opacity=0.90,
#             line=dict(width=0.8, color="white"),
#         ),
#         name=f"Gen {i} {elem}",
#         legendgroup=f"gen_{i}",
#         text=[elem] * len(mask),
#         textposition="top center",
#         hovertemplate="%{text}<br>(%{x:.2f}, %{y:.2f}, %{z:.2f})<extra>Gen %{fullData.name}</extra>",
#     ))

fig = go.Figure(traces)
fig.update_layout(
    title=dict(
        text=f"{td.name}  —  {pocket_name}<br>"
             f"<sup>pocket: {len(poc_coords)} atoms | GT ligand | {len(gen_mols)} generated</sup>",
        font=dict(size=13),
    ),
    scene=dict(
        xaxis_title="X (Å)", yaxis_title="Y (Å)", zaxis_title="Z (Å)",
        aspectmode="data",
    ),
    width=1000, height=720,
    legend=dict(itemsizing="constant", font=dict(size=9),
                groupclick="toggleitem"),
    margin=dict(l=0, r=0, t=60, b=0),
)
fig.show()

# print("Generated ligand atom types:", elems)

## 4. 2D Structure Grid

RDKit depictions of each generated ligand with key drug-likeness metrics.

In [ ]:
TARGETDIFF_ROOT = str(Path("..").resolve().parent / "targetdiff")
from utils.evaluation.scoring_func import get_chem   # from targetdiff
import sys
if TARGETDIFF_ROOT not in sys.path:
    sys.path.insert(0, TARGETDIFF_ROOT)
from utils.evaluation.scoring_func import get_chem


def mol_to_pil(mol: Chem.Mol, size=(280, 220)) -> Image.Image:
    """Render an RDKit mol to a PIL image."""
    drawer = rdMolDraw2D.MolDraw2DCairo(*size)
    drawer.drawOptions().addStereoAnnotation = True
    drawer.DrawMolecule(mol)
    drawer.FinishDrawing()
    return Image.open(BytesIO(drawer.GetDrawingText()))


def compute_metrics(mol: Chem.Mol) -> dict:
    c = get_chem(mol)
    return {
        "QED":      round(c["qed"], 3),
        "SA":       round(c["sa"], 2),
        "LogP":     round(c["logp"], 2),
        "Lipinski": int(c["lipinski"]),
        "HAtoms":   mol.GetNumHeavyAtoms(),
    }


# ── draw grid ─────────────────────────────────────────────────────────────────
n_cols = 4
n_rows = math.ceil(len(gen_mols) / n_cols)
mol_w, mol_h = 280, 220
panel_h = mol_h + 58          # extra space for text

fig_grid, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(n_cols * mol_w / 72, n_rows * panel_h / 72),
)
axes = np.array(axes).reshape(-1)   # flatten

for i, mol in enumerate(gen_mols):
    ax = axes[i]
    img = mol_to_pil(mol, size=(mol_w, mol_h))
    ax.imshow(img)
    ax.axis("off")

    m = compute_metrics(mol)
    smi = Chem.MolToSmiles(mol)
    color = palette[i % len(palette)]
    ax.set_title(
        f"Gen {i}\n"
        f"QED={m['QED']}  SA={m['SA']}  LogP={m['LogP']}\n"
        f"Lipinski={m['Lipinski']}/5  HA={m['HAtoms']}",
        fontsize=7,
        pad=3,
        bbox=dict(boxstyle="round,pad=0.3", facecolor=color, alpha=0.25, linewidth=0),
    )

# hide unused axes
for j in range(len(gen_mols), len(axes)):
    axes[j].set_visible(False)

fig_grid.suptitle(
    f"{td.name} — Generated Ligands  ({len(gen_mols)} mols)",
    fontsize=11, y=1.01,
)
fig_grid.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'utils'

## 5. Per-target Summary

Aggregated metrics across **all** target directories in the sample folder.

In [ ]:
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")


def tanimoto_diversity(mols):
    if len(mols) < 2:
        return float("nan")
    fps  = [Chem.RDKFingerprint(m) for m in mols]
    sims = [DataStructs.TanimotoSimilarity(fps[i], fps[j])
            for i in range(len(fps)) for j in range(i+1, len(fps))]
    return 1.0 - float(np.mean(sims))


rows = []
for td in target_dirs:
    sdf_path = td / "samples.sdf"
    if not sdf_path.exists():
        continue
    raw   = list(Chem.SDMolSupplier(str(sdf_path), sanitize=False))
    valid = [m for m in Chem.SDMolSupplier(str(sdf_path), sanitize=True)
             if m is not None and "." not in Chem.MolToSmiles(m)]

    if not valid:
        rows.append({"target": td.name, "n_total": len(raw), "n_valid": 0})
        continue

    smiles  = [Chem.MolToSmiles(m) for m in valid]
    chems   = [get_chem(m) for m in valid]
    rows.append({
        "target":     td.name,
        "n_total":    len(raw),
        "n_valid":    len(valid),
        "validity":   len(valid) / len(raw),
        "uniqueness": len(set(smiles)) / len(valid),
        "diversity":  tanimoto_diversity(valid),
        "qed":        np.mean([c["qed"]      for c in chems]),
        "sa":         np.mean([c["sa"]       for c in chems]),
        "logp":       np.mean([c["logp"]     for c in chems]),
        "lipinski":   np.mean([c["lipinski"] for c in chems]),
    })

# ── print summary table ───────────────────────────────────────────────────────
valid_rows = [r for r in rows if r.get("n_valid", 0) > 0]

def col_mean(key):
    vals = [r[key] for r in valid_rows if key in r and not np.isnan(r[key])]
    return np.mean(vals) if vals else float("nan")

keys = ["validity", "uniqueness", "diversity", "qed", "sa", "logp", "lipinski"]
print(f"Targets evaluated : {len(valid_rows)} / {len(rows)}")
print()
print(f"{'Metric':<14}  {'Mean':>8}")
print("-" * 25)
for k in keys:
    print(f"  {k:<12}  {col_mean(k):8.4f}")

NameError: name 'get_chem' is not defined

In [ ]:
# ── Bar plots of per-target metrics ──────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(valid_rows).set_index("target")

plot_keys = [("qed", "QED", "higher=better"),
             ("sa",  "SA",  "lower=better"),
             ("diversity", "Diversity", "higher=better"),
             ("validity",  "Validity",  "higher=better")]

fig_bar, axes = plt.subplots(2, 2, figsize=(14, 6))
for ax, (col, label, hint) in zip(axes.flatten(), plot_keys):
    vals = df[col].dropna()
    colors = [plt.cm.RdYlGn(v / vals.max()) if col != "sa"
              else plt.cm.RdYlGn(1 - v / 10) for v in vals]
    ax.bar(range(len(vals)), vals.values, color=colors, edgecolor="none")
    ax.axhline(vals.mean(), color="steelblue", ls="--", lw=1.2,
               label=f"mean={vals.mean():.3f}")
    ax.set_title(f"{label}  ({hint})", fontsize=9)
    ax.set_xlabel("target index", fontsize=8)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(range(len(vals)), fontsize=6, rotation=90)
    ax.legend(fontsize=8)

fig_bar.suptitle("Per-target metrics across all sampled pockets", fontsize=11)
fig_bar.tight_layout()
plt.show()

NameError: name 'valid_rows' is not defined

In [ ]:
# ── Interactive scatter: QED vs SA coloured by diversity ─────────────────────
df2 = df.reset_index()

fig_scatter = go.Figure(go.Scatter(
    x=df2["sa"],
    y=df2["qed"],
    mode="markers+text",
    text=df2["target"],
    textposition="top center",
    textfont=dict(size=7),
    marker=dict(
        size=df2["n_valid"] * 3,
        color=df2["diversity"],
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(title="Diversity", thickness=12, len=0.7),
        opacity=0.8,
        line=dict(width=0.5, color="white"),
    ),
    hovertemplate=(
        "<b>%{text}</b><br>"
        "SA=%{x:.2f}  QED=%{y:.3f}<br>"
        "Diversity=%{marker.color:.3f}<extra></extra>"
    ),
))
fig_scatter.update_layout(
    title="QED vs SA — bubble size = # valid mols, colour = diversity",
    xaxis_title="SA score  (lower = easier to synthesise)",
    yaxis_title="QED  (higher = more drug-like)",
    width=800, height=520,
)
fig_scatter.show()

NameError: name 'df' is not defined

## 3b. NGLView pocket surface + ligand

Interactive molecular view using **nglview**:
- pocket shown as a translucent surface
- GT ligand (if available) as ball-and-stick (+ optional ligand surface).

In [7]:
import nglview as nv
from pathlib import Path

if TARGET_IDX is None:
    raise ValueError("Set TARGET_IDX to a specific target above before running the NGLView cell.")

# Use the same target directory `td` as above
if 'td' not in globals():
    raise RuntimeError("Target directory `td` is not defined. Run the loading cell above first.")

target_dir: Path = td

# Pocket PDB
pdb_paths = sorted(target_dir.glob("*_pocket10.pdb"))
if not pdb_paths:
    raise FileNotFoundError(f"No pocket PDB found in {target_dir}")

pocket_pdb_path = pdb_paths[0]

# GT ligand SDF (non-samples.sdf); fall back to None if missing
gt_sdfs = [p for p in target_dir.glob("*.sdf") if p.name != "samples.sdf"]
ligand_sdf_path = gt_sdfs[0] if gt_sdfs else None
print(ligand_sdf_path)

print(f"Pocket PDB : {pocket_pdb_path}")
print(f"Ligand SDF : {ligand_sdf_path if ligand_sdf_path is not None else 'None (GT ligand not found)'}")

view = nv.NGLWidget(height="600px", width="600px")

# Pocket as surface
view.add_component(str(pocket_pdb_path))
view.clear_representations(component=0)
view.add_representation(
    "surface",
    component=0,
    opacity=0.6,
    color="element",
    surfaceType="av",
)

# Ligand as molecules (ball+stick + optional surface), if present
if ligand_sdf_path is not None:
    view.add_component(str(ligand_sdf_path))
    lig_comp = 1
    view.clear_representations(component=lig_comp)
    view.add_representation(
        "ball+stick",
        component=lig_comp,
        colorScheme="element",
        multipleBond="symmetric",
    )
    view.add_representation(
        "surface",
        component=lig_comp,
        opacity=0.2,
        color="white",
        surfaceType="vws",
    )

view.center()
view

/home/shpark/prj-ligand/Voxbind/voxbind/exps/exp_sig0.9/samples/res/target_00/BSD_ASPTE_1_130_0__2z3h_A_rec_1wn6_bst_lig_tt_docked_3.sdf
Pocket PDB : /home/shpark/prj-ligand/Voxbind/voxbind/exps/exp_sig0.9/samples/res/target_00/BSD_ASPTE_1_130_0__2z3h_A_rec_1wn6_bst_lig_tt_docked_3_pocket10.pdb
Ligand SDF : /home/shpark/prj-ligand/Voxbind/voxbind/exps/exp_sig0.9/samples/res/target_00/BSD_ASPTE_1_130_0__2z3h_A_rec_1wn6_bst_lig_tt_docked_3.sdf


NGLWidget()

## 3c. NGLView pocket surface + generated ligand

Same pocket surface as above, but with the **first generated ligand** (`samples.sdf`) shown as ball-and-stick.

In [11]:
import tempfile, os
from rdkit.Chem import AllChem

if not gen_mols:
    raise RuntimeError("No generated molecules found – run the loading cell first.")

# Write the first generated mol to a temp SDF so nglview can load it
gen_mol = gen_mols[0]
gen_sdf_path = os.path.join(tempfile.gettempdir(), "gen_ligand_0.sdf")
writer = Chem.SDWriter(gen_sdf_path)
writer.write(gen_mol)
writer.close()

print(f"Generated ligand written to: {gen_sdf_path}")
print(f"  Atoms : {gen_mol.GetNumAtoms()}")
print(f"  SMILES: {Chem.MolToSmiles(gen_mol)}")

view_gen = nv.NGLWidget(height="600px", width="600px")

# Pocket as surface (same as 3b)
view_gen.add_component(str(pocket_pdb_path))
view_gen.clear_representations(component=0)
view_gen.add_representation(
    "surface",
    component=0,
    opacity=0.1,
    color="element",
    surfaceType="av",
)

# Generated ligand as ball+stick
view_gen.add_component(gen_sdf_path)
view_gen.clear_representations(component=1)
view_gen.add_representation(
    "ball+stick",
    component=1,
    colorScheme="element",
    multipleBond="symmetric",
)
view_gen.add_representation(
    "surface",
    component=1,
    opacity=0.6,
    color="white",
    surfaceType="vws",
)

view_gen.center()
view_gen

Generated ligand written to: /tmp/gen_ligand_0.sdf
  Atoms : 23
  SMILES: C=CC=C1CC=C(C)C(O)=C1[C@@H]1C=C(C=O)[C@@H](CCC)ON1O


NGLWidget()